<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week3_unsupervised/Week3_Lesson6_Workshop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛒 IB CS — Week 3 · Lesson 6 (Workshop)
## Workshop "Unsupervised in Action"
### A4.3.4 + A4.3.5 + A4.3.10 in practice

> ⏱ Duration: 2 academic hours.
> 💻 Platform: Google Colab
> 🎯 Goal: complete **3 unsupervised tasks** on realistic datasets and compare supervised models with cross-validation.

---

### 📋 Workshop plan

| Part | Topic | Time |
|---|---|---|
| 1 | **K-means** customer segmentation + Elbow method | 25 min |
| 2 | **DBSCAN** on non-spherical data | 20 min |
| 3 | **Apriori** market basket analysis | 30 min |
| 4 | **Model comparison** using cross-validation | 25 min |
| 5 | IB-style conclusion | 10 min |

---

### ⚙️ Library setup

> If `mlxtend` is not installed in Colab, run in the first cell: `!pip install mlxtend`


In [ ]:
# Install mlxtend if needed for Apriori
# !pip install mlxtend --quiet

# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

np.random.seed(2027)
sns.set_style('whitegrid')
print("✅ All libraries are ready")


## Part 1 · Customer Segmentation with K-means

### Scenario
> A clothing store wants to segment customers using 2 features:
> - **Annual income** (thousand $)
> - **Spending score** (1-100; tendency to buy)

The goal is to create **different marketing strategies** for each group.


In [ ]:
# Synthetic dataset: 5 customer types
n_per_group = 50
customers = []
labels_true = []

# Group 1: high income + high spending ("Champions")
customers.append(np.random.normal([85, 80], [10, 8], (n_per_group, 2)))
# Group 2: low income + low spending ("Budget")
customers.append(np.random.normal([25, 25], [8, 8], (n_per_group, 2)))
# Group 3: high income + low spending ("Misers")
customers.append(np.random.normal([85, 25], [10, 8], (n_per_group, 2)))
# Group 4: low income + high spending ("Risky")
customers.append(np.random.normal([25, 80], [8, 8], (n_per_group, 2)))
# Group 5: middle group ("Average")
customers.append(np.random.normal([55, 55], [10, 10], (n_per_group, 2)))

X_cust = np.vstack(customers)
df_cust = pd.DataFrame(X_cust, columns=['Annual Income (k$)', 'Spending Score'])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df_cust['Annual Income (k$)'], df_cust['Spending Score'],
           c='gray', s=50, alpha=0.6, edgecolor='black')
ax.set_xlabel('Annual Income (k$)', fontsize=11)
ax.set_ylabel('Spending Score (1-100)', fontsize=11)
ax.set_title('Customers BEFORE clustering — labels are unknown',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"📊 Total customers: {len(df_cust)}")


### 🎯 TRY IT #1 — How many clusters do you see?

Before running the algorithm, **guess**:
- How many natural customer groups are there?
- Where are the centroids, approximately?

> 💡 This matters: in a real IA, justify K using both Elbow and domain knowledge.


### Step 1 · Elbow Method — choosing K


In [ ]:
# Try K from 1 to 10
ks = range(1, 11)
inertias = []
silhouettes = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cust)
    inertias.append(km.inertia_)
    if k > 1:
        silhouettes.append(silhouette_score(X_cust, km.labels_))
    else:
        silhouettes.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ks, inertias, 'o-', linewidth=2, markersize=10, color='steelblue')
axes[0].axvline(5, color='red', linestyle='--', alpha=0.7, label='Elbow at K=5')
axes[0].set_xlabel('K (number of clusters)', fontsize=11)
axes[0].set_ylabel('Inertia (sum of squared distances)', fontsize=11)
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ks, silhouettes, 'o-', linewidth=2, markersize=10, color='orange')
axes[1].axvline(5, color='red', linestyle='--', alpha=0.7, label='Best at K=5')
axes[1].set_xlabel('K', fontsize=11)
axes[1].set_ylabel('Silhouette Score (higher = better)', fontsize=11)
axes[1].set_title('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f"💎 Best K by Elbow: about 5, where inertia starts falling more slowly")
print(f"💎 Best K by Silhouette: {np.nanargmax(silhouettes) + 1}")


> 💎 **IA secret:** show **both charts**: Elbow + Silhouette. That gives **2 independent** justifications for choosing K, which examiners value.

### Step 2 · Final segmentation with K=5


In [ ]:
# Final model
best_k = 5
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cust['cluster'] = km_final.fit_predict(X_cust)

# Segment names: business interpretation
segment_names = {}
for c in range(best_k):
    centroid = km_final.cluster_centers_[c]
    income, spending = centroid
    if income > 60 and spending > 60:
        segment_names[c] = 'Champions (high income, high spending)'
    elif income < 40 and spending < 40:
        segment_names[c] = 'Budget (low income, low spending)'
    elif income > 60 and spending < 40:
        segment_names[c] = 'Misers (high income, low spending)'
    elif income < 40 and spending > 60:
        segment_names[c] = 'Risky (low income, high spending)'
    else:
        segment_names[c] = 'Average (middle)'

# Visualisation
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#FF6B6B', '#4ECDC4', '#FFD93D', '#6BCB77', '#A78BFA']
for c in range(best_k):
    mask = df_cust['cluster'] == c
    ax.scatter(df_cust.loc[mask, 'Annual Income (k$)'],
               df_cust.loc[mask, 'Spending Score'],
               c=colors[c], s=60, alpha=0.7, edgecolor='black',
               label=segment_names[c])
ax.scatter(km_final.cluster_centers_[:,0], km_final.cluster_centers_[:,1],
           marker='X', s=300, c='black', edgecolor='white', linewidth=2,
           label='Centroids')
ax.set_xlabel('Annual Income (k$)', fontsize=11)
ax.set_ylabel('Spending Score', fontsize=11)
ax.set_title(f'Customer Segmentation (K-means, K={best_k})',
             fontsize=12, fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
plt.tight_layout(); plt.show()

# Table: strategy for each segment
print("\n=== Business strategies for each segment ===")
strategies = pd.DataFrame({
    'Segment': [segment_names[c] for c in range(best_k)],
    'Size': [(df_cust['cluster']==c).sum() for c in range(best_k)],
    'Strategy': [
        'VIP programme, exclusive products',
        'Discounts, budget collections',
        'Premium marketing, brand image',
        'Credit options, careful upselling',
        'Standard marketing'
    ]
})
print(strategies.to_string(index=False))


> 💎 **IA secret:** after clustering, always give a **business interpretation**. Just saying "5 clusters" is weak. Naming them Champions, Budget, Misers, Risky, Average and giving a strategy for each is much stronger.


## Part 2 · DBSCAN — when K-means fails

### Scenario
> Analysing tourist GPS tracks in Barcelona. Groups form along streets, so they are elongated, and there are **outliers** from isolated tourists.


In [ ]:
# Generate data: 2 half-moons + noise
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
# Add outliers
outliers = np.random.uniform(-1.5, 2.5, (15, 2))
X_with_noise = np.vstack([X_moons, outliers])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original data
axes[0].scatter(X_with_noise[:,0], X_with_noise[:,1], c='gray', s=30, alpha=0.7)
axes[0].set_title('Original data with outliers', fontsize=11)

# K-means: failure
km = KMeans(n_clusters=2, random_state=42, n_init=10)
km_labels = km.fit_predict(X_with_noise)
axes[1].scatter(X_with_noise[:,0], X_with_noise[:,1], c=km_labels, cmap='viridis',
                s=30, alpha=0.7)
axes[1].scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1],
                marker='X', s=200, c='red', edgecolor='black')
axes[1].set_title('K-means: cuts half-moons\n(outliers forced into clusters)',
                  fontsize=11, fontweight='bold', color='darkred')

# DBSCAN: success
_db = DBSCAN(eps=0.2, min_samples=5)
db_labels = _db.fit_predict(X_with_noise)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)

# Black points = noise
mask_noise = db_labels == -1
mask_clusters = ~mask_noise
axes[2].scatter(X_with_noise[mask_clusters,0], X_with_noise[mask_clusters,1],
                c=db_labels[mask_clusters], cmap='viridis', s=30, alpha=0.7)
axes[2].scatter(X_with_noise[mask_noise,0], X_with_noise[mask_noise,1],
                c='black', s=80, marker='x', linewidth=2, label=f'Noise ({n_noise})')
axes[2].set_title(f'DBSCAN: {n_clusters} clusters + {n_noise} noise',
                  fontsize=11, fontweight='bold', color='darkgreen')
axes[2].legend()

plt.tight_layout(); plt.show()


### 🎯 TRY IT #2 — Experiment with DBSCAN parameters

In the code below, change `eps` and `min_samples`. What happens?
- `eps=0.1` → too many points become noise / clusters fragment
- `eps=0.5` → too much merges into one cluster
- `min_samples=20` → more points become noise


In [ ]:
# Experiment with DBSCAN parameters
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

configs = [(0.1, 5), (0.2, 5), (0.4, 5)]
for ax, (eps, min_s) in zip(axes, configs):
    db = DBSCAN(eps=eps, min_samples=min_s)
    labels = db.fit_predict(X_with_noise)
    n_clust = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)

    mask_noise = labels == -1
    ax.scatter(X_with_noise[~mask_noise,0], X_with_noise[~mask_noise,1],
               c=labels[~mask_noise], cmap='viridis', s=30, alpha=0.7)
    ax.scatter(X_with_noise[mask_noise,0], X_with_noise[mask_noise,1],
               c='black', s=80, marker='x', linewidth=2)
    ax.set_title(f'eps={eps}, min_samples={min_s}\n→ {n_clust} clusters, {n_noise} noise',
                 fontsize=11)

plt.suptitle('Effect of eps on DBSCAN', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


## Part 3 · Apriori — Market Basket Analysis

### Scenario
> A grocery store wants to understand **which products are bought together** so it can:
> - Place them close together on shelves
> - Offer bundle discounts
> - Recommend items in an online basket


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Synthetic transactions: 50 purchases
transactions = [
    ['milk', 'bread', 'butter'],
    ['milk', 'bread', 'eggs'],
    ['milk', 'bread'],
    ['bread', 'butter'],
    ['milk', 'butter', 'eggs'],
    ['bread', 'butter', 'jam'],
    ['milk', 'bread', 'butter'],
    ['eggs', 'cheese'],
    ['milk', 'bread', 'butter', 'eggs'],
    ['bread', 'jam'],
    ['milk', 'eggs'],
    ['bread', 'butter', 'cheese'],
    ['milk', 'bread', 'jam'],
    ['cheese', 'wine'],
    ['milk', 'bread', 'butter'],
    ['bread', 'butter', 'eggs'],
    ['milk', 'cheese'],
    ['bread', 'jam', 'butter'],
    ['wine', 'cheese', 'bread'],
    ['milk', 'bread', 'eggs', 'cheese'],
    ['bread', 'butter'],
    ['milk', 'bread'],
    ['milk', 'butter'],
    ['eggs', 'milk'],
    ['bread', 'butter', 'cheese'],
    ['wine', 'cheese'],
    ['milk', 'bread', 'butter'],
    ['bread', 'butter', 'eggs'],
    ['milk', 'bread', 'jam'],
    ['cheese', 'wine', 'bread'],
    ['milk', 'eggs', 'butter'],
    ['bread', 'butter'],
    ['milk', 'bread', 'butter', 'eggs'],
    ['jam', 'bread', 'butter'],
    ['milk', 'cheese', 'wine'],
    ['bread', 'butter', 'jam'],
    ['eggs', 'milk', 'bread'],
    ['cheese', 'wine', 'bread', 'butter'],
    ['milk', 'bread'],
    ['butter', 'jam', 'bread'],
    ['milk', 'bread', 'butter'],
    ['eggs', 'butter'],
    ['cheese', 'milk'],
    ['wine', 'cheese', 'bread', 'butter'],
    ['milk', 'bread', 'jam', 'butter'],
    ['bread', 'butter', 'eggs', 'cheese'],
    ['milk', 'butter', 'jam'],
    ['cheese', 'bread', 'butter'],
    ['milk', 'eggs', 'cheese'],
    ['bread', 'butter', 'milk', 'eggs'],
]

print(f"📊 Total transactions: {len(transactions)}")

# One-hot encoding
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_ary, columns=te.columns_)
print(f"📊 Unique products: {len(te.columns_)}")
print("\nFirst 5 transactions (one-hot):")
print(df_trans.head().astype(int))


In [ ]:
# Step 1: find frequent itemsets with min_support = 0.2 (20%)
frequent_itemsets = apriori(df_trans, min_support=0.2, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print("=== Frequent Itemsets (support >= 0.20) ===")
print(frequent_itemsets.sort_values(['length', 'support'], ascending=[True, False]).to_string(index=False))


In [ ]:
# Step 2: create association rules
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5)

# Show the most interesting rules, sorted by lift
print("=== Top 10 Association Rules (sorted by Lift) ===")
display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
print(rules[display_cols].sort_values('lift', ascending=False).head(10).round(3).to_string(index=False))


In [ ]:
# Visualisation: scatter plot Support vs Confidence, size = Lift
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(rules['support'], rules['confidence'],
                     s=rules['lift']*300, alpha=0.5,
                     c=rules['lift'], cmap='RdYlGn',
                     edgecolor='black')
ax.set_xlabel('Support', fontsize=11)
ax.set_ylabel('Confidence', fontsize=11)
ax.set_title('Association Rules: Support × Confidence (size/colour = Lift)',
             fontsize=12, fontweight='bold')
plt.colorbar(scatter, label='Lift')
ax.grid(alpha=0.3)

# Label the top 3 rules
top3 = rules.nlargest(3, 'lift')
for _, row in top3.iterrows():
    label = f"{list(row['antecedents'])} → {list(row['consequents'])}"
    ax.annotate(label, (row['support'], row['confidence']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=9, fontweight='bold')

plt.tight_layout(); plt.show()


### 🎯 TRY IT #3 — Business interpretation of rules

Looking at the top rules, answer:

1. Which 2 products have the **highest lift**? What does that mean for the store?
2. Which rule is most **reliable**, meaning highest confidence?
3. Which rule has the **highest support**, meaning it happens often?

> 💎 In a real exam (Section B), you may be shown a rule table and asked: *"Which rule should the marketing team prioritize? Justify."* Look at **lift** for association strength and **support** to check that it happens often enough.


## Part 4 · Model Selection using Cross-Validation

> We return to **supervised learning** from last week, but now compare models **properly** with CV.

### Scenario
> Compare **3 models** for classification using **5-fold cross-validation** and multiple metrics.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

# Binary classification: 500 records, 8 features
X_clf, y_clf = make_classification(n_samples=500, n_features=8, n_informative=5,
                                    n_redundant=2, random_state=42)

# 3 models for comparison: KNN uses a pipeline for normalisation
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()),
                                      ('lr', LogisticRegression(max_iter=1000))]),
    'Decision Tree (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'KNN (k=7)': Pipeline([('scaler', StandardScaler()),
                            ('knn', KNeighborsClassifier(n_neighbors=7))]),
}

# Cross-validate across all 4 metrics at once
metrics = ['accuracy', 'precision', 'recall', 'f1']
results = []

for name, model in models.items():
    cv = cross_validate(model, X_clf, y_clf, cv=5, scoring=metrics)
    row = {'Model': name}
    for m in metrics:
        scores = cv[f'test_{m}']
        row[f'{m}_mean'] = scores.mean()
        row[f'{m}_std']  = scores.std()
    results.append(row)

results_df = pd.DataFrame(results)
print("=== Cross-Validation Results (5-fold) ===\n")
for _, row in results_df.iterrows():
    print(f"{row['Model']}:")
    for m in metrics:
        print(f"  {m:10s} = {row[f'{m}_mean']:.4f} ± {row[f'{m}_std']:.4f}")
    print()


In [ ]:
# Visualisation: bar chart with error bars
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results_df))
width = 0.2
colors = ['#FF6B6B', '#4ECDC4', '#FFD93D', '#6BCB77']

for i, m in enumerate(metrics):
    means = results_df[f'{m}_mean'].values
    stds  = results_df[f'{m}_std'].values
    ax.bar(x + i*width, means, width, yerr=stds, capsize=4,
           label=m, color=colors[i], edgecolor='black')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.set_ylabel('Score (5-fold CV)', fontsize=11)
ax.set_title('Model Comparison — Cross-Validation Results\n(error bars = std across folds)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()


### 🎯 TRY IT #4 — Choose the best model

From the chart:
1. Which model is **best by F1**?
2. Which model has the **smallest spread** (std), so is more reliable?
3. If the business priority is **high recall**, such as medicine, which model would you choose?
4. If the priority is **interpretability**, which model would you choose?

> 💎 **Secret:** in an IA or Section B answer, **never** choose a model using only one metric. Your justification should consider **at least 3 criteria**: F1, std, interpretability, and context.


## Part 5 · IB-style conclusion

### 📝 Template for an IA report on unsupervised learning

```
1. PROBLEM TYPE: Unsupervised learning (no labels in the data)
   - Goal: customer segmentation / pattern discovery

2. ALGORITHM CHOICE:
   - K-means for known K + spherical clusters
   - DBSCAN for arbitrary shapes + handling outliers
   - Hierarchical for hierarchical data when K is unknown
   - Apriori for co-occurrence patterns

3. HYPERPARAMETER SELECTION:
   - K using Elbow method + Silhouette score
   - ε and minPts for DBSCAN through experiments
   - Minimum support and confidence for Apriori through business context

4. EVALUATION:
   - Visualisation: scatter plots coloured by cluster
   - Silhouette score
   - Business interpretation of clusters, with names

5. RECOMMENDATIONS:
   - Specific actions for each segment
   - Trade-offs between algorithms
```

---

### 💎 Final workshop secrets

1. **Elbow + Silhouette together** give a reliable K choice.
2. **DBSCAN sees what K-means cannot**: non-spherical clusters and outliers.
3. **Lift > 1** is the key association-rule metric because it shows association strength.
4. **Business names for segments** can strengthen an IA.
5. **Cross-validation with std** (error bars) is a gold standard.
6. **mlxtend** is the standard Python library for Apriori.

---

### 🏠 Homework (see `Week3_HW2_Practice.ipynb`)

> Full mini-IA: customer segmentation + market basket + model selection report.
